In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error


In [3]:
base = "e:/Agriculture Data"

# TEST
df_test_b1 = pd.read_csv(f"{base}/Bilinear Tables/TestB_08_08_24.csv")
df_test_c1 = pd.read_csv(f"{base}/Cubic Tables/TestC_08_08_24.csv")
df_test_n1 = pd.read_csv(f"{base}/Nearest Tables/TestN_08_08_24.csv")

df_test_b2 = pd.read_csv(f"{base}/Bilinear Tables/TestB_08_15_24.csv")
df_test_c2 = pd.read_csv(f"{base}/Cubic Tables/TestC_08_15_24.csv")
df_test_n2 = pd.read_csv(f"{base}/Nearest Tables/TestN_08_15_24.csv")

df_test_b3 = pd.read_csv(f"{base}/Bilinear Tables/TestB_09_13_24.csv")
df_test_c3 = pd.read_csv(f"{base}/Cubic Tables/TestC_09_13_24.csv")
df_test_n3 = pd.read_csv(f"{base}/Nearest Tables/TestN_09_13_24.csv")

df_test_b4 = pd.read_csv(f"{base}/Bilinear Tables/TestB_09_20_24.csv")
df_test_c4 = pd.read_csv(f"{base}/Cubic Tables/TestC_09_20_24.csv")
df_test_n4 = pd.read_csv(f"{base}/Nearest Tables/TestN_09_20_24.csv")

df_test_b5 = pd.read_csv(f"{base}/Bilinear Tables/TestB_10_04_24.csv")
df_test_c5 = pd.read_csv(f"{base}/Cubic Tables/TestC_10_04_24.csv")
df_test_n5 = pd.read_csv(f"{base}/Nearest Tables/TestN_10_04_24.csv")

df_test_b6 = pd.read_csv(f"{base}/Bilinear Tables/TestB_10_18_24.csv")
df_test_c6 = pd.read_csv(f"{base}/Cubic Tables/TestC_10_18_24.csv")
df_test_n6 = pd.read_csv(f"{base}/Nearest Tables/TestN_10_18_24.csv")

df_test_b7 = pd.read_csv(f"{base}/Bilinear Tables/TestB_11_08_24.csv")
df_test_c7 = pd.read_csv(f"{base}/Cubic Tables/TestC_11_08_24.csv")
df_test_n7 = pd.read_csv(f"{base}/Nearest Tables/TestN_11_08_24.csv")

df_test_b8 = pd.read_csv(f"{base}/Bilinear Tables/TestB_11_15_24.csv")
df_test_c8 = pd.read_csv(f"{base}/Cubic Tables/TestC_11_15_24.csv")
df_test_n8 = pd.read_csv(f"{base}/Nearest Tables/TestN_11_15_24.csv")


# TRAIN
df_train_b1 = pd.read_csv(f"{base}/Bilinear Tables/TrainB_08_08_24.csv")
df_train_c1 = pd.read_csv(f"{base}/Cubic Tables/TrainC_08_08_24.csv")
df_train_n1 = pd.read_csv(f"{base}/Nearest Tables/TrainN_08_08_24.csv")

df_train_b2 = pd.read_csv(f"{base}/Bilinear Tables/TrainB_08_15_24.csv")
df_train_c2 = pd.read_csv(f"{base}/Cubic Tables/TrainC_08_15_24.csv")
df_train_n2 = pd.read_csv(f"{base}/Nearest Tables/TrainN_08_15_24.csv")

df_train_b3 = pd.read_csv(f"{base}/Bilinear Tables/TrainB_09_13_24.csv")
df_train_c3 = pd.read_csv(f"{base}/Cubic Tables/TrainC_09_13_24.csv")
df_train_n3 = pd.read_csv(f"{base}/Nearest Tables/TrainN_09_13_24.csv")

df_train_b4 = pd.read_csv(f"{base}/Bilinear Tables/TrainB_09_20_24.csv")
df_train_c4 = pd.read_csv(f"{base}/Cubic Tables/TrainC_09_20_24.csv")
df_train_n4 = pd.read_csv(f"{base}/Nearest Tables/TrainN_09_20_24.csv")

df_train_b5 = pd.read_csv(f"{base}/Bilinear Tables/TrainB_10_04_24.csv")
df_train_c5 = pd.read_csv(f"{base}/Cubic Tables/TrainC_10_04_24.csv")
df_train_n5 = pd.read_csv(f"{base}/Nearest Tables/TrainN_10_04_24.csv")

df_train_b6 = pd.read_csv(f"{base}/Bilinear Tables/TrainB_10_18_24.csv")
df_train_c6 = pd.read_csv(f"{base}/Cubic Tables/TrainC_10_18_24.csv")
df_train_n6 = pd.read_csv(f"{base}/Nearest Tables/TrainN_10_18_24.csv")

df_train_b7 = pd.read_csv(f"{base}/Bilinear Tables/TrainB_11_08_24.csv")
df_train_c7 = pd.read_csv(f"{base}/Cubic Tables/TrainC_11_08_24.csv")
df_train_n7 = pd.read_csv(f"{base}/Nearest Tables/TrainN_11_08_24.csv")

df_train_b8 = pd.read_csv(f"{base}/Bilinear Tables/TrainB_11_15_24.csv")
df_train_c8 = pd.read_csv(f"{base}/Cubic Tables/TrainC_11_15_24.csv")
df_train_n8 = pd.read_csv(f"{base}/Nearest Tables/TrainN_11_15_24.csv")

In [30]:
def add_features(df, date_str, weather_df):
    # --- DATE FEATURES ---
    df['date'] = pd.to_datetime(date_str)
    df['day_of_year'] = df['date'].dt.dayofyear
    df['month'] = df['date'].dt.month

    # cyclical encoding (important)
    df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    # --- WEATHER FEATURES ---
    weather_row = weather_df[weather_df['date'] == date_str].iloc[0]
    
    weather_cols = [
        'tavg_c','tmin_c','tmax_c',
        'precip_mm','srad_mj_m2',
        'wind_max_kmh','wind_gusts_kmh',
        'rh_pct','et0_mm','vpd_max_kpa'
    ]

    for col in weather_cols:
        if col in weather_row.index:
            df[col] = weather_row[col]

    # --- INDICES (Landsat 8) ---
    B2 = df['Band2']  # Blue
    B3 = df['Band3']  # Green
    B4 = df['Band4']  # Red
    B5 = df['Band5']  # NIR
    B7 = df['Band7']  # SWIR2

    # Green Normalized Difference Vegetation Index
    df['L_GNDVI'] = (B5 - B3) / (B5 + B3 + 1e-6)

    # Normalized Difference Water Index
    df['L_NDWI'] = (B3 - B5) / (B3 + B5 + 1e-6)

    # Soil Adjusted Vegetation Index
    df['L_SAVI'] = ((B5 - B4) / (B5 + B4 + 0.5)) * 1.5

    # Enhanced Vegetation Index
    df['L_EVI'] = 2.5 * (B5 - B4) / (B5 + 6*B4 - 7.5*B2 + 1 + 1e-6)

    # Ratio Vegetation Index
    df['L_RVI'] = B5 / (B4 + 1e-6)

    # Visible Atmospherically Resistant Index (uses only visible bands)
    df['L_VARI'] = (B3 - B4) / (B3 + B4 - B2 + 1e-6)

    # Normalized Burn Ratio (used for vegetation stress and dryness)
    df['L_NBR'] = (B5 - B7) / (B5 + B7 + 1e-6)

    return df

In [31]:
date_map = {
    1: "2024-08-08",
    2: "2024-08-15",
    3: "2024-09-13",
    4: "2024-09-20",
    5: "2024-10-04",
    6: "2024-10-18",
    7: "2024-11-08",
    8: "2024-11-15",
}

weather_df = pd.read_csv("ndvi_features_Walnut_CA.csv")

for i in range(1, 9):
    # TEST
    df_test_b = globals()[f'df_test_b{i}']
    df_test_c = globals()[f'df_test_c{i}']
    df_test_n = globals()[f'df_test_n{i}']

    globals()[f'df_test_b{i}'] = add_features(df_test_b, date_map[i], weather_df)
    globals()[f'df_test_c{i}'] = add_features(df_test_c, date_map[i], weather_df)
    globals()[f'df_test_n{i}'] = add_features(df_test_n, date_map[i], weather_df)

    # TRAIN
    df_train_b = globals()[f'df_train_b{i}']
    df_train_c = globals()[f'df_train_c{i}']
    df_train_n = globals()[f'df_train_n{i}']

    globals()[f'df_train_b{i}'] = add_features(df_train_b, date_map[i], weather_df)
    globals()[f'df_train_c{i}'] = add_features(df_train_c, date_map[i], weather_df)
    globals()[f'df_train_n{i}'] = add_features(df_train_n, date_map[i], weather_df)

In [36]:
df_train_b1.head()

,X,Y,UAV_NDVI,L_NDVI,Band1,Band2,Band3,Band4,Band5,Band6,...,tavg_c,tmin_c,tmax_c,precip_mm,srad_mj_m2,wind_max_kmh,wind_gusts_kmh,rh_pct,et0_mm,vpd_max_kpa
0,425082.243001,3.767240e+06,0.303958,0.2466,10171,10343,11160,11256,18731,17776,...,25.8,20.8,32.0,0.0,27.67,12.6,36.4,59,5.96,3.0
1,425082.258021,3.767240e+06,0.285604,0.2466,10171,10343,11160,11256,18731,17776,...,25.8,20.8,32.0,0.0,27.67,12.6,36.4,59,5.96,3.0
2,425082.273041,3.767240e+06,0.280383,0.2466,10171,10343,11160,11256,18731,17776,...,25.8,20.8,32.0,0.0,27.67,12.6,36.4,59,5.96,3.0
3,425082.288061,3.767240e+06,0.301367,0.2466,10171,10343,11160,11256,18731,17776,...,25.8,20.8,32.0,0.0,27.67,12.6,36.4,59,5.96,3.0
4,425082.303081,3.767240e+06,0.303625,0.2466,10171,10343,11160,11256,18731,17776,...,25.8,20.8,32.0,0.0,27.67,12.6,36.4,59,5.96,3.0


In [33]:
test_nearest = []
test_bilinear = []
test_cubic = []

train_nearest = []
train_bilinear = []
train_cubic = []

for var_name, var_value in list(globals().items()):
    if isinstance(var_value, pd.DataFrame):
        if var_name.startswith('df_test_'):
            if var_name[-2] == 'n':
                test_nearest.append(var_value)
            elif var_name[-2] == 'b':
                test_bilinear.append(var_value)
            elif var_name[-2] == 'c':
                test_cubic.append(var_value)

        elif var_name.startswith('df_train_'):
            if var_name[-2] == 'n':
                train_nearest.append(var_value)
            elif var_name[-2] == 'b':
                train_bilinear.append(var_value)
            elif var_name[-2] == 'c':
                train_cubic.append(var_value)

print(f"Nearest  — Train: {len(train_nearest)}, Test: {len(test_nearest)}")
print(f"Bilinear — Train: {len(train_bilinear)}, Test: {len(test_bilinear)}")
print(f"Cubic    — Train: {len(train_cubic)}, Test: {len(test_cubic)}")

Nearest  — Train: 8, Test: 8
Bilinear — Train: 8, Test: 8
Cubic    — Train: 8, Test: 8


In [34]:
resamples = {
    'nearest':  (train_nearest,  test_nearest),
    'bilinear': (train_bilinear, test_bilinear),
    'cubic':    (train_cubic,    test_cubic),
}

for name, (train_list, test_list) in resamples.items():
    # total rows across all DataFrames
    train_rows = sum(df.shape[0] for df in train_list)
    test_rows  = sum(df.shape[0] for df in test_list)

    total = train_rows + test_rows

    print(f"\n{name}:")
    print(f"Train rows: {train_rows} ({train_rows/total:.2%})")
    print(f"Test  rows: {test_rows} ({test_rows/total:.2%})")


nearest:
Train rows: 8995059 (57.29%)
Test  rows: 6706589 (42.71%)

bilinear:
Train rows: 8995059 (57.29%)
Test  rows: 6706589 (42.71%)

cubic:
Train rows: 8995059 (58.83%)
Test  rows: 6294838 (41.17%)


In [35]:
train_list, test_list = resamples['bilinear']

# combine into full dataframes
train_df = pd.concat(train_list, ignore_index=True)
test_df  = pd.concat(test_list, ignore_index=True)

# --- split test in half spatially (by x) ---
threshold = test_df['X'].median()
test_half1 = test_df[test_df['X'] < threshold]
test_half2 = test_df[test_df['X'] >= threshold]

# move half into training
train_df = pd.concat([train_df, test_half2], ignore_index=True)
test_df  = test_half1

# --- define target + features ---
target_col = 'UAV_NDVI'

feature_cols = [
    # raw bands
    'Band1','Band2','Band3','Band4','Band5','Band6','Band7',

    # vegetation indices (Landsat-derived)
    'L_NDVI','L_GNDVI','L_NDWI','L_SAVI','L_EVI','L_RVI','L_VARI','L_NBR',

    # time features
    'sin_doy','cos_doy',

    # optional quality features
    'AEROSOL_Band_1','PIXEL_Band_1','RADSAT_Band_1'
]

# ensure columns exist (prevents crashes)
feature_cols = [c for c in feature_cols if c in train_df.columns]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

# --- train model ---
model = LinearRegression()
model.fit(X_train, y_train)

# --- predict ---
y_pred = model.predict(X_test)

# --- evaluate ---
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

MemoryError: Unable to allocate 25.6 MiB for an array with shape (3353466, 1) and data type int64